# DS2002 · Midterm Checkpoint

**Lab — 2026-10-23 · Fall 2026**  

---

## Lab 09 — Midterm checkpoint, week 1 of 3

**What you submit: the link to your team's main repo. Only that.** One submission per team. Do not upload separate files.

Put this notebook in the repo (the `notebooks/` folder is fine) and push it, so everything we check is in one place.

**You do not have to fill this all out.** It is a checkpoint, not a finished project. Skip any section you have not started and write "not started" next to it. What we must see is:

1. **Your charter**, committed to the repo
2. **Any progress** you have made, with code that ran and output you can point at

Start with the team setup, then work down the same eight things we listed at Monday's kickoff, as far as you have gotten. This is graded on evidence, not on intentions.

The purpose is to catch a stalled team while there is still time to unstall it. If something is not working, say so here plainly. A checkpoint that honestly reports "the API join is not working and here is the error" is worth more than one that claims progress that does not exist, and it gets you help on Monday.

### Team setup

This part is required. Fill it in once for the whole team.

You need:

- Your team name, and every member with a GitHub handle (2 to 4 people)
- A link to your team's **main repo**. This is the one link you submit.
- A link to `CHARTER.md` in that repo, the page on GitHub, not a file on someone's laptop

Nice to have, and also in your charter: this week's notebook owner, your meeting time, and the five worst problems in the data from the profile you ran on Monday. Be specific: "`unit_price` is stored as text and some values start with `$`" beats "prices are messy."

In [1]:
team_setup = {
    'team_name': 'M&M',

'members': [
    {'name': 'Maryam', 'github': 'marykahmed77-cmd'},
    {'name': 'Mariam', 'github': 'mariams11-cmd'},  # I'm not actually sure what Mariam's github username is, and it's late at night to ask. I will ask as soon as possible tomorrow!
    ],
    'repo_url': 'https://github.com/marykahmed77-cmd/walmart-hurricane-project/tree/main',                           # your main team repo: https://github.com/...
    'charter_url': 'https://github.com/marykahmed77-cmd/walmart-hurricane-project/blob/main/CHARTER.md',                        # link to CHARTER.md in that repo
    'notebook_owner_this_week': 'Maryam',           # nice to have
    'meeting_time': 'Once a week call on Sundays; text every other day/ whenever needed',                       # nice to have, e.g. 'Tue and Thu 7pm'
    'five_worst_problems': ['Unresponsiveness from teammates; not validating; incorrectly unloading files to github; procrastination; incorrectly doing one step that affects the entire notebook'],              # nice to have, five specific problems
}

def check_team(t, quiet=False):
    def real(v):
        return isinstance(v, str) and v.strip() != '' and not v.startswith('TODO')
    required = {
        'team name': real(t['team_name']),
        '2 to 4 members, each with a name and GitHub handle':
            2 <= len(t['members']) <= 4 and all(real(m['name']) and real(m['github']) for m in t['members']),
        'main repo link': real(t['repo_url']) and t['repo_url'].startswith('https://github.com/'),
        'charter link points at CHARTER.md': real(t['charter_url']) and t['charter_url'].endswith('CHARTER.md'),
    }
    nice = {
        'notebook owner': real(t['notebook_owner_this_week']),
        'meeting time': real(t['meeting_time']),
        'five specific problems': len(t['five_worst_problems']) >= 5 and all(real(p) for p in t['five_worst_problems']),
    }
    if not quiet:
        print('REQUIRED')
        for label, good in required.items():
            print('  PASS' if good else '  FAIL', '-', label)
        print('NICE TO HAVE')
        for label, good in nice.items():
            print('  done' if good else '  not yet', '-', label)
        print()
        print('team setup complete:', all(required.values()))
    return all(required.values())

check_team(team_setup)

REQUIRED
  PASS - team name
  PASS - 2 to 4 members, each with a name and GitHub handle
  PASS - main repo link
  PASS - charter link points at CHARTER.md
NICE TO HAVE
  done - notebook owner
  done - meeting time
  not yet - five specific problems

team setup complete: True


True

Now show that the charter is actually committed. This is required. Run the next cell and leave its output in the notebook. Point `REPO_PATH` at your local clone of the team repo. If it prints a line like `a1b2c3d Add team charter`, you are done. If it prints "no commits found," the charter is not in the repo yet, so commit it and push before you submit.

In [2]:
import subprocess
!git clone https://github.com/marykahmed77-cmd/walmart-hurricane-project.git
REPO_PATH = '/content/walmart-hurricane-project'      # your local clone of the team repo

result = subprocess.run(['git', '-C', REPO_PATH, 'log', '--oneline', '--', 'CHARTER.md'],
                        capture_output=True, text=True)
print(result.stdout.strip() or 'no commits found for CHARTER.md')

fatal: destination path 'walmart-hurricane-project' already exists and is not an empty directory.
5873aa5 adding team charter


### 1 — Your cleaning pipeline as functions

**TODO:** paste the cleaning code you have written so far. It should be functions, not loose cells, and it should cover at minimum: duplicates, `unit_price` to numeric (it comes in as text, some with a `$`), the negative `quantity` values with your decision applied, `timestamp` parsed, and `store_id` cleaned up so it will join to the store file.

Print the row count before and after. We want to see how many rows your decisions removed.

In [3]:

import pandas as pd

# Load Walmart Transactions dataset from GitHub
url = "https://raw.githubusercontent.com/marykahmed77-cmd/walmart-hurricane-project/main/data/walmart_transactions.csv"

df = pd.read_csv(url)

# Preserve the original dataset
df_original = df.copy()

# Verify the dataset loaded
print("Original dataset shape:", df_original.shape)
display(df_original.head())


Original dataset shape: (25562, 8)


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-13T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,fl-412,2024-09-02 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-06 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-30 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-15 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82


In [4]:
# TODO
import pandas as pd

# Standard category names
CATEGORY_MAP = {
    "grocery": "Grocery",
    "groceries": "Grocery",
    "household": "Household",
    "household essentials": "Household",
    "electronics": "Electronics",
    "alcohol": "Alcohol",
    "alcoholic beverages": "Alcohol",
    "snacks": "Snacks",
    "emergency": "Emergency",
    "emergency supplies": "Emergency",
    "entertainment": "Entertainment",
    "games & entertainment": "Entertainment",
    "health": "Health",
    "health & wellness": "Health",
    "breakfast": "Breakfast",
    "breakfast foods": "Breakfast",
    "baby": "Baby",
    "baby products": "Baby"
}

def clean_transactions(raw):
    # Work on a copy so original data remains unchanged
    df_clean = raw.copy()
    decision_log = []

    def log_decision(issue, action, reason, count, status="Applied"):
        decision_log.append({
            "Issue": issue,
            "Cleaning Decision": action,
            "Justification": reason,
            "Rows Affected": int(count),
            "Status": status
        })

    # 1. Remove exact duplicate records
    duplicate_count = df_clean.duplicated().sum()
    df_clean = df_clean.drop_duplicates().copy()

    log_decision(
        "Exact duplicate records",
        "Remove exact duplicate rows",
        "Prevent double-counting identical transactions",
        duplicate_count
    )

    # 2. Standardize store IDs
    old_store_ids = df_clean["store_id"].copy()

    df_clean["store_id"] = (
        df_clean["store_id"]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL(\d{3})$", r"FL-\1", regex=True)
    )

    changed_store_ids = (
        old_store_ids.astype("string") != df_clean["store_id"]
    ).sum()

    log_decision(
        "Inconsistent store IDs",
        "Standardize capitalization, spaces and hyphens",
        "Create consistent identifiers for merging with store data",
        changed_store_ids
    )

    # 3. Convert unit prices to numeric
    original_prices = df_clean["unit_price"].astype("string")

    formatted_prices = original_prices.str.contains(
        r"[$,]", regex=True, na=False
    ).sum()

    df_clean["unit_price"] = pd.to_numeric(
        original_prices
        .str.strip()
        .str.replace(r"[$,]", "", regex=True),
        errors="coerce"
    )

    log_decision(
        "Unit prices stored as text",
        "Remove dollar signs and commas; convert to numeric",
        "Allow accurate price and revenue calculations",
        formatted_prices
    )

    # Check for prices that could not be converted
    invalid_prices = df_clean["unit_price"].isna().sum()

    log_decision(
        "Unparseable unit prices",
        "Flag missing or unparseable numeric prices",
        "Avoid introducing incorrect zero-dollar transactions",
        invalid_prices,
        "Review" if invalid_prices else "Checked: none"
    )

    # 4. Standardize timestamps
    df_clean["timestamp"] = pd.to_datetime(
        df_clean["timestamp"],
        format="mixed",
        errors="coerce"
    )

    df_clean["missing_timestamp"] = df_clean["timestamp"].isna()

    log_decision(
        "Mixed timestamp formats",
        "Convert valid dates to a consistent datetime format",
        "Support daily sales and hurricane timeline analysis",
        df_clean["timestamp"].notna().sum()
    )

    log_decision(
        "Missing timestamps",
        "Retain records and flag missing timestamps",
        "Preserve transactions without inventing dates",
        df_clean["missing_timestamp"].sum(),
        "Flagged"
    )

    # 5. Standardize product categories
    original_categories = df_clean["category"].copy()

    normalized_categories = (
        df_clean["category"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    # Check that every category has a mapping
    unknown_categories = (
        set(normalized_categories.dropna().unique())
        - set(CATEGORY_MAP)
    )

    if unknown_categories:
        raise ValueError(
            f"Unmapped categories: {unknown_categories}"
        )

    df_clean["category"] = normalized_categories.map(CATEGORY_MAP)

    changed_categories = (
        original_categories.astype("string") !=
        df_clean["category"]
    ).sum()

    log_decision(
        "Inconsistent category labels",
        "Standardize capitalization and equivalent categories",
        "Prevent equivalent categories from being counted separately",
        changed_categories
    )

    # 6. Flag negative quantities
    df_clean["negative_quantity"] = df_clean["quantity"] < 0

    log_decision(
        "Negative transaction quantities",
        "Retain original quantities and flag negative values",
        "Negative quantities may represent returns or adjustments",
        df_clean["negative_quantity"].sum(),
        "Flagged"
    )

    # 7. Preserve SKUs until product catalog verification
    log_decision(
        "Inconsistent product names and SKUs",
        "Preserve original identifiers pending catalog review",
        "Avoid merging distinct products or package sizes",
        df_clean.shape[0],
        "Pending catalog review"
    )

    return df_clean, pd.DataFrame(decision_log)


# Run the cleaning function
df_clean, decision_log = clean_transactions(df_original)

print("Cleaning complete!")
print("Original rows:", len(df_original))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df_original) - len(df_clean))

Cleaning complete!
Original rows: 25562
Cleaned rows: 25532
Rows removed: 30


### 2 — Your decision log

**TODO:** every cleaning decision, what it did, and how many rows it touched. If your teammates disagree with one of these, note the disagreement — that is useful information, not a problem to hide.

In [5]:
# TODO: print or display your decision log
display(decision_log)
print("Total decisions documented:", len(decision_log))

,Issue,Cleaning Decision,Justification,Rows Affected,Status
0,Exact duplicate records,Remove exact duplicate rows,Prevent double-counting identical transactions,30,Applied
1,Inconsistent store IDs,"Standardize capitalization, spaces and hyphens",Create consistent identifiers for merging with...,4345,Applied
2,Unit prices stored as text,Remove dollar signs and commas; convert to num...,Allow accurate price and revenue calculations,3026,Applied
3,Unparseable unit prices,Flag missing or unparseable numeric prices,Avoid introducing incorrect zero-dollar transa...,0,Checked: none
4,Mixed timestamp formats,Convert valid dates to a consistent datetime f...,Support daily sales and hurricane timeline ana...,24648,Applied
5,Missing timestamps,Retain records and flag missing timestamps,Preserve transactions without inventing dates,884,Flagged
6,Inconsistent category labels,Standardize capitalization and equivalent cate...,Prevent equivalent categories from being count...,18744,Applied
7,Negative transaction quantities,Retain original quantities and flag negative v...,Negative quantities may represent returns or a...,130,Flagged
8,Inconsistent product names and SKUs,Preserve original identifiers pending catalog ...,Avoid merging distinct products or package sizes,25532,Pending catalog review


Total decisions documented: 9


### 3 — SKU consolidation progress

**TODO:** how many distinct SKUs did you start with, how many canonical products do you have now, and what is your largest consolidated group? Show the mapping for at least one product family. Pop-Tarts is a good one to start with.

You do not need this finished — you need it started and defensible.

In [6]:
# TODO

original_skus = df_original["sku"].nunique()
print("Original distinct SKUs:", original_skus)

poptarts = df_clean[
    df_clean["product_name"].str.contains(
        "pop-tart|pop tart", case=False, na=False
    )
]

poptart_mapping = (
    poptarts[["sku", "product_name"]]
    .drop_duplicates()
    .sort_values("sku")
)

print("\nPop-Tarts SKU mapping candidates:")
display(poptart_mapping)
print("Distinct Pop-Tarts SKUs:", poptarts["sku"].nunique())

Original distinct SKUs: 45

Pop-Tarts SKU mapping candidates:


,sku,product_name
1177,POPTART-STRAW,Pop-Tarts Strawberry
853,POPTART-STRAW,Pop Tarts - Strawberry
669,POPTART-STRAW,Pop-Tarts Strawberry
52,POPTART-STRAW,Strawberry Pop-Tart
123,POPTART-STRAW,pop-tarts strawberry
360,PT-12,pop-tarts strawberry
264,PT-12,Pop-Tarts Strawberry
1125,PT-12,Pop-Tarts Strawberry
227,PT-12,Strawberry Pop-Tart
47,PT-12,Pop Tarts - Strawberry


Distinct Pop-Tarts SKUs: 4


### 4 — One working weather API call

**TODO:** show the call and the parsed DataFrame. It must include the retry logic and a `source` column. Say which timezone the weather timestamps are in and how you will line them up with the sales timestamps. If it is not working, paste the actual error and what you have tried.

In [7]:
# TODO

not started

### 5 — One chart

**TODO:** daily order volume across your date range. It does not need to be presentation quality yet, but it does need labeled axes and a title.

Look at it before you submit. Does the shape make sense? An early chart is the fastest way to catch a cleaning bug — if your daily volume has a day at zero or a day at ten times everything else, that is usually your pipeline, not the hurricane.

In [8]:
# TODO

not started

### 6 — Self-check

Fill in what you have and leave the rest. **The required items are the team setup and the charter.** Everything else is progress we want to see, not a pass or fail. The checks are shape checks; they cannot verify your analysis is right, only that you have gotten far enough to have one.

In [10]:
required = {
    'team_setup_complete': check_team(team_setup, quiet=True),   # from the team setup section
    'charter_committed': True,         # TODO: True once CHARTER.md shows in the git log above
}

progress = {
    'rows_after_cleaning': 25532,        # an integer
    'decisions_logged': 9,           # how many entries in your log
    'canonical_products': 45,         # distinct products after consolidation
    'weather_api_rows': None,           # rows returned by your API call
    'notebook_runs_clean': True,       # True after Restart and Run All
}

def is_done(value):
    return not (value is None or value is False
                or (isinstance(value, str) and value.startswith('TODO')))

def check(required, progress):
    print('REQUIRED')
    for label, value in required.items():
        print('  PASS' if is_done(value) else '  FAIL', '-', label, '=', value)
    print('PROGRESS (fill in what you have)')
    for label, value in progress.items():
        print('  done' if is_done(value) else '  not yet', '-', label, '=', value)
    ok = all(is_done(v) for v in required.values())
    print()
    print('required items complete:', ok)
    print('progress items done:', sum(is_done(v) for v in progress.values()), 'of', len(progress))
    return ok

check(required, progress)

REQUIRED
  PASS - team_setup_complete = True
  PASS - charter_committed = True
PROGRESS (fill in what you have)
  done - rows_after_cleaning = 25532
  done - decisions_logged = 9
  done - canonical_products = 45
  not yet - weather_api_rows = None
  done - notebook_runs_clean = True

required items complete: True
progress items done: 4 of 5


True

### 7 — Risk and plan

Two questions, answered concretely.

**a)** What is the single most likely thing to go wrong in the next two weeks? Name a specific technical or team problem, not "running out of time."

**b)** What are you doing about it this week? Name who is doing it and by when.

**Biggest risk:** _Unresponsiveness. Also kind of worried that I may have done the cleaning wrong._

**Plan, with an owner and a date:** _Mariam will do weather API adn will work on visualizations._

### 8 — Where each question stands

**TODO:** mark each of the five questions. These owners must match the question owners table in your charter. Be honest — this is the section I read first.


| Question | Status | Who owns it |
|---|---|---|
| Q1 demand surge | Not started | Mariam |
| Q2 SKU consolidation | In progress | Maryam |
| Q3 weather correlation | Not started | Mariam |
| Q4 store and geography | Not started | Mariam |
| Q5 signal or artifact | Not started | Maryam |


At the end of week 1, "not started" on Q4 and Q5 is expected. "Not started" on all five means come find me on Monday.